# Build derived TorNet modeling manifests

Build a validated modeling manifest from the preserved raw
annual audit without rescanning or modifying the source archive.

The official test set is preserved exactly. Explicitly documented
training exclusions resolve confirmed train/test event leakage.


In [1]:
%pip install -q xarray netCDF4 pandas pyarrow


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%pip install --force-reinstall --no-deps "/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.3-py3-none-any.whl"


Processing ./drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.3-py3-none-any.whl
  Attempting uninstall: tornet-detection
    Found existing installation: tornet-detection 0.1.2
    Uninstalling tornet-detection-0.1.2:
      Successfully uninstalled tornet-detection-0.1.2


In [4]:
from pathlib import Path
import json

import pandas as pd

import tornado_detection

from tornado_detection.data.modeling import (
    ModelingExclusion,
    build_modeling_manifest,
    write_modeling_manifest_artifacts,
)

assert tornado_detection.__version__ == "0.1.3"

print(
    "tornado_detection package version:",
    tornado_detection.__version__,
)
print(
    "Loaded tornado_detection from:",
    tornado_detection.__file__,
)


tornado_detection package version: 0.1.3
Loaded tornado_detection from: /usr/local/lib/python3.12/dist-packages/tornado_detection/__init__.py


## 2018 leakage resolution

Preserve all official test files and exclude only the seven
training files participating in confirmed event `784979` leakage.


In [5]:
RAW_DIRECTORY = Path(
    "/content/drive/MyDrive/TorNet_Backup/manifests/v1/2018"
)

OUTPUT_DIRECTORY = Path(
    "/content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018"
)

EXPECTED_DIMENSIONS = {
    "time": 4,
    "sweep": 2,
    "azimuth": 120,
    "range": 240,
    "lims": 2,
}

EXCLUSION_MEMBERS = (
    (
        "train/2018/"
        "WRN_180913_215731_KMHX_1080869n_U3.nc"
    ),
    (
        "train/2018/"
        "WRN_180913_220345_KMHX_1080869n_F9.nc"
    ),
    (
        "train/2018/"
        "WRN_180913_220958_KMHX_1080869n_U0.nc"
    ),
    (
        "train/2018/"
        "WRN_180913_221610_KMHX_1080869n_U0.nc"
    ),
    (
        "train/2018/"
        "WRN_180913_222224_KMHX_1080869n_T9.nc"
    ),
    (
        "train/2018/"
        "WRN_180913_222922_KMHX_1080869n_N5.nc"
    ),
    (
        "train/2018/"
        "WRN_180913_224208_KMHX_1080869n_N8.nc"
    ),
)

EXCLUSIONS = [
    ModelingExclusion(
        archive_member=archive_member,
        expected_event_id="784979",
        expected_episode_id="130979",
        reason=(
            "Official train/test event leakage"
        ),
        resolution=(
            "Preserve all official test files and "
            "exclude the overlapping training members"
        ),
    )
    for archive_member in EXCLUSION_MEMBERS
]

assert len(EXCLUSION_MEMBERS) == 7
assert len(set(EXCLUSION_MEMBERS)) == 7

print("Raw manifest:", RAW_DIRECTORY)
print("Modeling output:", OUTPUT_DIRECTORY)
print("Configured exclusions:", len(EXCLUSIONS))


Raw manifest: /content/drive/MyDrive/TorNet_Backup/manifests/v1/2018
Modeling output: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018
Configured exclusions: 7


In [6]:
modeling_build = build_modeling_manifest(
    RAW_DIRECTORY,
    exclusions=EXCLUSIONS,
    expected_dimensions=EXPECTED_DIMENSIONS,
)

ledger = modeling_build.exclusion_ledger
validation = modeling_build.validation
manifest = modeling_build.manifest

display(ledger)
display(validation.checks)

print(
    "Event groups crossing official splits:"
)
display(validation.event_split_overlap)

print(
    "Episode groups crossing official splits "
    "(informational):"
)
display(validation.episode_split_overlap)

assert manifest.year == 2018
assert (
    modeling_build.source_terminal_marker
    == "_INVALID.json"
)

assert len(ledger) == 7
assert (
    ledger["archive_member"].tolist()
    == list(EXCLUSION_MEMBERS)
)
assert set(ledger["split"]) == {"train"}
assert set(ledger["category"]) == {"WRN"}
assert set(
    ledger["event_id"].astype(str)
) == {"784979"}
assert set(
    ledger["episode_id"].astype(str)
) == {"130979"}
assert set(ledger["radar_site"]) == {"KMHX"}

assert int(
    ledger["removed_frame_count"].sum()
) == 28
assert int(
    ledger[
        "removed_positive_frame_count"
    ].sum()
) == 0

assert len(
    manifest.file_manifest
) == 17_871
assert len(
    manifest.frame_manifest
) == 71_484
assert (
    manifest.netcdf_member_count
    == 17_871
)

assert validation.all_required_passed
assert validation.event_split_overlap.empty

print(
    "PASS: 2018 modeling manifest validated "
    "before writing"
)


,manifest_schema_version,year,archive_member,file_id,split,category,event_id,episode_id,radar_site,removed_frame_count,removed_positive_frame_count,reason,resolution
0,1.0.0,2018,train/2018/WRN_180913_215731_KMHX_1080869n_U3.nc,5320afa0ade042220043cb72b548757938dffa06e4c63d...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...
1,1.0.0,2018,train/2018/WRN_180913_220345_KMHX_1080869n_F9.nc,cc15b4a5731d5a82edd42eb0dbf92420cd8494da94738e...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...
2,1.0.0,2018,train/2018/WRN_180913_220958_KMHX_1080869n_U0.nc,9c3811d2540e3131a6a59578d1ccfc5244664d2cc8a396...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...
3,1.0.0,2018,train/2018/WRN_180913_221610_KMHX_1080869n_U0.nc,78655029934cd57fcd2be081aba8603369c5a5c9c335fe...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...
4,1.0.0,2018,train/2018/WRN_180913_222224_KMHX_1080869n_T9.nc,a7f157236c1fe395c4bd284de904849dc453680a7df84a...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...
5,1.0.0,2018,train/2018/WRN_180913_222922_KMHX_1080869n_N5.nc,e96afb8788ac9e24d475f79e86d88b780dfdfd4556588c...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...
6,1.0.0,2018,train/2018/WRN_180913_224208_KMHX_1080869n_N8.nc,c9083dc7db65d4ef4c5596ed38fb279ae0e6c6c623b435...,train,WRN,784979,130979,KMHX,4,0,Official train/test event leakage,Preserve all official test files and exclude t...


,check,required,passed,observed,expected,detail
0,build_errors,True,True,0,0,
1,file_row_count,True,True,17871,17871,
2,frame_row_count,True,True,71484,71484,
3,unique_archive_members,True,True,17871,17871,
4,unique_file_ids,True,True,17871,17871,
5,unique_frame_ids,True,True,71484,71484,
6,frame_rows_match_file_frame_counts,True,True,0,0,
7,frame_label_sums_match_file_manifest,True,True,0,0,
8,frame_indices_are_contiguous,True,True,0,0,
9,expected_frames_per_file,True,True,0,0,Expected 4 frames for every file


Event groups crossing official splits:


,event_group_id,splits_json,file_count,archive_members_json


Episode groups crossing official splits (informational):


,episode_id,splits_json,file_count,archive_members_json
0,129109,"[""test"",""train""]",53,"[""test/2018/NUL_180807_220954_KCYS_773839s_B0...."
1,130037,"[""test"",""train""]",13,"[""test/2018/NUL_181006_212144_KPBZ_784163s_T9...."
2,130979,"[""test"",""train""]",94,"[""test/2018/WRN_180914_000435_KRAX_1080870n_A1..."


PASS: 2018 modeling manifest validated before writing


In [7]:
artifacts = write_modeling_manifest_artifacts(
    modeling_build,
    OUTPUT_DIRECTORY,
    overwrite=False,
)

for artifact_name, artifact_path in sorted(
    artifacts.items()
):
    print(f"- {artifact_name}: {artifact_path}")

required_artifacts = {
    "file_manifest.parquet",
    "frame_manifest.parquet",
    "schema_summary.csv",
    "build_errors.csv",
    "validation_checks.csv",
    "event_split_overlap.csv",
    "episode_split_overlap.csv",
    "category_frame_summary.csv",
    "manifest_summary.json",
    "exclusion_ledger.csv",
    "modeling_summary.json",
    "_SUCCESS.json",
}

assert required_artifacts.issubset(
    artifacts.keys()
)
assert "_INVALID.json" not in artifacts

summary = json.loads(
    (
        OUTPUT_DIRECTORY
        / "modeling_summary.json"
    ).read_text()
)

assert summary["year"] == 2018
assert summary["status"] == "valid"
assert (
    summary["source_raw_terminal_marker"]
    == "_INVALID.json"
)
assert summary["source_file_row_count"] == 17_878
assert summary["source_frame_row_count"] == 71_512
assert summary["modeling_file_row_count"] == 17_871
assert summary["modeling_frame_row_count"] == 71_484
assert summary["excluded_file_count"] == 7
assert summary["excluded_frame_count"] == 28
assert (
    summary["excluded_positive_frame_count"]
    == 0
)
assert summary["event_split_overlap_count"] == 0
assert (
    summary["all_required_validations_passed"]
    is True
)

print(
    "PASS: 2018 modeling manifest built "
    "and written"
)


- _SUCCESS.json: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/_SUCCESS.json
- build_errors.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/build_errors.csv
- category_frame_summary.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/category_frame_summary.csv
- episode_split_overlap.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/episode_split_overlap.csv
- event_split_overlap.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/event_split_overlap.csv
- exclusion_ledger.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/exclusion_ledger.csv
- file_manifest.parquet: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/file_manifest.parquet
- frame_manifest.parquet: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/frame_manifest.parquet
- manifest_summary.json: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2018/manifest_summary.json
- mod